In [8]:
import pandas as pd
import numpy as np

In [9]:
df = pd.read_csv('redo_merged_dataset.csv')

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2970 entries, 0 to 2969
Data columns (total 61 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   id                                     2970 non-null   int64  
 1   is_weekend                             2970 non-null   bool   
 2   day_in_study                           2970 non-null   int64  
 3   phase                                  2970 non-null   object 
 4   lh                                     2970 non-null   float64
 5   estrogen                               2970 non-null   float64
 6   flow_volume                            2578 non-null   object 
 7   flow_color                             2578 non-null   object 
 8   appetite                               2970 non-null   object 
 9   exerciselevel                          2970 non-null   object 
 10  headaches                              2970 non-null   object 
 11  cram

In [11]:
df.columns

Index(['id', 'is_weekend', 'day_in_study', 'phase', 'lh', 'estrogen',
       'flow_volume', 'flow_color', 'appetite', 'exerciselevel', 'headaches',
       'cramps', 'sorebreasts', 'fatigue', 'sleepissue', 'moodswing', 'stress',
       'foodcravings', 'indigestion', 'bloating', 'calories_daily',
       'steps_daily', 'bpm_mean', 'bpm_min', 'bpm_max', 'bpm_std',
       'temp_diff_from_baseline_mean', 'temp_diff_from_baseline_min',
       'temp_diff_from_baseline_max', 'temp_diff_from_baseline_std',
       'glucose_mean', 'glucose_min', 'glucose_max', 'glucose_std',
       'overall_score', 'composition_score', 'revitalization_score',
       'duration_score', 'nightly_temperature', 'duration',
       'minutestofallasleep', 'minutesasleep', 'minutesawake',
       'minutesafterwakeup', 'timeinbed', 'efficiency', 'duration_exercise',
       'calories_exercise', 'heartrate_exercise', 'activity_type',
       'in_default_zone_3', 'in_default_zone_2', 'in_default_zone_1',
       'below_default_zo

In [12]:
import pandas as pd

df = df.sort_values(["id", "day_in_study"]).reset_index(drop=True)

def assign_cycle(sub):
    sub = sub.copy().reset_index(drop=True)
 
    is_new_cycle = (sub["phase"] == "Menstrual") & (sub["phase"].shift(1) != "Menstrual")
   
    sub["cycle"] = is_new_cycle.cumsum()
    
    return sub

df = df.groupby("id", group_keys=False).apply(assign_cycle)

df = df[df["cycle"] > 0]


/var/tmp/ipykernel_4534/634064563.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("id", group_keys=False).apply(assign_cycle)


In [13]:
df["cycle_length"] = (
    df.groupby(["id", "cycle"])["day_in_study"]
            .transform("count")
)

In [14]:
df.head()

,id,is_weekend,day_in_study,phase,lh,estrogen,flow_volume,flow_color,appetite,exerciselevel,...,below_default_zone_1,birth_year,gender,ethnicity,education,sexually_active,self_report_menstrual_health_literacy,age_of_first_menarche,cycle,cycle_length
19,2,False,20,Menstrual,1.0,65.7,Not at all,Not at all,Low,Low,...,1355.0,1995,Woman,East Asian,"Bachelor's degree (e.g. BA, BS)",Yes,High,13,1,31
20,2,True,21,Menstrual,4.8,81.6,Moderate,Bright Red,High,Very High,...,831.0,1995,Woman,East Asian,"Bachelor's degree (e.g. BA, BS)",Yes,High,13,1,31
21,2,True,22,Menstrual,5.8,90.2,Somewhat Light,Dark Brown / Dark Red,Moderate,Low,...,1358.0,1995,Woman,East Asian,"Bachelor's degree (e.g. BA, BS)",Yes,High,13,1,31
22,2,False,23,Menstrual,4.5,60.0,Light,Dark Brown / Dark Red,Low,Low,...,1430.0,1995,Woman,East Asian,"Bachelor's degree (e.g. BA, BS)",Yes,High,13,1,31
23,2,False,24,Follicular,3.5,60.0,Spotting / Very Light,Dark Brown / Dark Red,Moderate,Low,...,1408.0,1995,Woman,East Asian,"Bachelor's degree (e.g. BA, BS)",Yes,High,13,1,31


In [45]:
df['flow_volume'].value_counts()

flow_volume
Not at all               1969
Spotting / Very Light     175
Moderate                  119
Light                     114
Somewhat Light             80
Somewhat Heavy             54
Heavy                      47
Very Heavy                 20
Name: count, dtype: int64

In [46]:
mapping_flow_volume = {
    "Not at all": 0.0,
    "Spotting / Very Light": 1.0,
    "Light": 1.0,
    "Somewhat Light": 1.0,
    "Moderate": 2.0,
    "Somewhat Heavy": 2.0,
    "Heavy": 3.0,
    "Very Heavy": 3.0
}

df["flow_volume_score"] = df["flow_volume"].map(mapping_flow_volume)


In [47]:
#df['flow_color'].value_counts()

In [48]:
df['appetite'].value_counts()

appetite
Moderate      1443
Low            708
High           544
Very Low       181
Very High       89
Not at all       5
Name: count, dtype: int64

In [49]:
mapping = {
    "Not at all": 0.0,
    "Very Low": 1.0,
    "Low": 1.0,
    "Moderate": 2.0,
    "High": 3.0,
    "Very High": 3.0
}

df["appetite_score"] = df["appetite"].map(mapping_appetite)

In [50]:
df['exerciselevel'].value_counts()

exerciselevel
Low           1004
Moderate       873
Very Low       722
High           308
Very High       55
Not at all       8
Name: count, dtype: int64

In [51]:
df["exerciselevel_score"] = df["exerciselevel"].map(mapping)

In [52]:
df['headaches'].value_counts()

headaches
Not at all         933
Very Low/Little    866
Low                440
Moderate           403
High               233
Very High           95
Name: count, dtype: int64

In [53]:
mapping = {
    "Not at all": 0.0,
    "Very Low/Little": 1.0,
    "Low": 1.0,
    "Moderate": 2.0,
    "High": 3.0,
    "Very High": 3.0
}

In [54]:
df["headaches_score"] = df["headaches"].map(mapping)

In [55]:
df['cramps'].value_counts()

cramps
Not at all         1404
Very Low/Little     916
Moderate            232
Low                 231
High                125
Very High            62
Name: count, dtype: int64

In [56]:
df["cramps_score"] = df["cramps"].map(mapping)

In [57]:
df['sorebreasts'].value_counts()

sorebreasts
Not at all         1523
Very Low/Little     899
Low                 255
Moderate            207
High                 57
Very High            29
Name: count, dtype: int64

In [58]:
df["sorebreasts_score"] = df["sorebreasts"].map(mapping)

In [59]:
df['fatigue'].value_counts()

fatigue
Moderate           780
High               619
Low                537
Very Low/Little    445
Not at all         330
Very High          259
Name: count, dtype: int64

In [60]:
df["fatigue_score"] = df["fatigue"].map(mapping)

In [61]:
df['sleepissue'].value_counts()

sleepissue
Low                655
Very Low/Little    603
Moderate           585
Not at all         555
High               357
Very High          215
Name: count, dtype: int64

In [62]:
df["sleepissue_score"] = df["sleepissue"].map(mapping)

In [63]:
df['moodswing'].value_counts()

moodswing
Not at all         896
Very Low/Little    736
Moderate           503
Low                471
High               274
Very High           90
Name: count, dtype: int64

In [64]:
df["moodswing_score"] = df["moodswing"].map(mapping)

In [65]:
df['stress'].value_counts()

stress
Moderate           979
Low                536
High               527
Very Low/Little    328
Not at all         320
Very High          280
Name: count, dtype: int64

In [66]:
df["stress_score"] = df["stress"].map(mapping)

In [67]:
df['foodcravings'].value_counts()

foodcravings
Not at all         735
Very Low/Little    580
Low                568
Moderate           534
High               387
Very High          166
Name: count, dtype: int64

In [68]:
df["foodcravings_score"] = df["foodcravings"].map(mapping)

In [69]:
df['indigestion'].value_counts()

indigestion
Not at all         1005
Very Low/Little     695
Low                 494
Moderate            470
High                219
Very High            87
Name: count, dtype: int64

In [70]:
df["indigestion_score"] = df["indigestion"].map(mapping)

In [71]:
df['bloating'].value_counts()

bloating
Not at all         901
Very Low/Little    611
Low                540
Moderate           508
High               316
Very High           94
Name: count, dtype: int64

In [72]:
df["bloating_score"] = df["bloating"].map(mapping)

In [73]:
df.head()

,id,is_weekend,day_in_study,phase,lh,estrogen,flow_volume,flow_color,appetite,exerciselevel,...,headaches_score,cramps_score,sorebreasts_score,fatigue_score,sleepissue_score,moodswing_score,stress_score,foodcravings_score,indigestion_score,bloating_score
0,2,True,1,Fertility,2.9,78.5,Not at all,Not at all,Moderate,Very Low,...,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0
1,2,False,2,Fertility,3.2,126.9,Not at all,Not at all,Moderate,Very Low,...,0.0,0.0,0.0,2.0,0.0,0.0,2.0,0.0,1.0,0.0
2,2,False,3,Fertility,5.7,187.5,Not at all,Not at all,Low,Very Low,...,0.0,0.0,0.0,1.0,1.0,0.0,3.0,0.0,1.0,0.0
3,2,False,4,Fertility,26.4,220.3,Not at all,Not at all,Moderate,Moderate,...,0.0,0.0,0.0,3.0,2.0,2.0,3.0,0.0,0.0,0.0
4,2,False,5,Fertility,19.5,287.7,Not at all,Not at all,Moderate,Low,...,0.0,0.0,0.0,3.0,3.0,2.0,3.0,0.0,0.0,0.0


In [74]:
df_cleaned = df.copy()

In [77]:
df_cleaned.columns

Index(['id', 'is_weekend', 'day_in_study', 'phase', 'lh', 'estrogen',
       'flow_volume', 'flow_color', 'appetite', 'exerciselevel', 'headaches',
       'cramps', 'sorebreasts', 'fatigue', 'sleepissue', 'moodswing', 'stress',
       'foodcravings', 'indigestion', 'bloating', 'calories_daily',
       'steps_daily', 'bpm_mean', 'bpm_min', 'bpm_max', 'bpm_std',
       'temp_diff_from_baseline_mean', 'temp_diff_from_baseline_min',
       'temp_diff_from_baseline_max', 'temp_diff_from_baseline_std',
       'glucose_mean', 'glucose_min', 'glucose_max', 'glucose_std',
       'overall_score', 'composition_score', 'revitalization_score',
       'duration_score', 'nightly_temperature', 'duration',
       'minutestofallasleep', 'minutesasleep', 'minutesawake',
       'minutesafterwakeup', 'timeinbed', 'efficiency', 'duration_exercise',
       'calories_exercise', 'heartrate_exercise', 'activity_type',
       'in_default_zone_3', 'in_default_zone_2', 'in_default_zone_1',
       'below_default_zo

In [78]:
df_cleaned.drop(columns = ['flow_volume', 'appetite', 'exerciselevel', 'headaches',
       'cramps', 'sorebreasts', 'fatigue', 'sleepissue', 'moodswing', 'stress',
       'foodcravings', 'indigestion', 'bloating'], inplace = True)

In [81]:
dict_path = '/home/jupyter/Feature Selection/df.csv'
df.to_csv(dict_path, index=False)
print(f"\n✓ Data dictionary saved to: {dict_path}")
print(f"\nPreview:")
print(df.head(10).to_string(index=False))


✓ Data dictionary saved to: /home/jupyter/Feature Selection/df.csv

Preview:
 id  is_weekend  day_in_study     phase   lh  estrogen flow_volume flow_color appetite exerciselevel       headaches          cramps     sorebreasts  fatigue      sleepissue       moodswing    stress foodcravings indigestion        bloating  calories_daily  steps_daily  bpm_mean  bpm_min  bpm_max   bpm_std  temp_diff_from_baseline_mean  temp_diff_from_baseline_min  temp_diff_from_baseline_max  temp_diff_from_baseline_std  glucose_mean  glucose_min  glucose_max  glucose_std  overall_score  composition_score  revitalization_score  duration_score  nightly_temperature   duration  minutestofallasleep  minutesasleep  minutesawake  minutesafterwakeup  timeinbed  efficiency  duration_exercise  calories_exercise  heartrate_exercise activity_type  in_default_zone_3  in_default_zone_2  in_default_zone_1  below_default_zone_1  birth_year gender  ethnicity                       education sexually_active self_report_menstr

In [80]:
dict_path = '/home/jupyter/Feature Selection/df_cleaned.csv'
df_cleaned.to_csv(dict_path, index=False)
print(f"\n✓ Data dictionary saved to: {dict_path}")
print(f"\nPreview:")
print(df_cleaned.head(10).to_string(index=False))


✓ Data dictionary saved to: /home/jupyter/Feature Selection/df_cleaned.csv

Preview:
 id  is_weekend  day_in_study     phase   lh  estrogen flow_color  calories_daily  steps_daily  bpm_mean  bpm_min  bpm_max   bpm_std  temp_diff_from_baseline_mean  temp_diff_from_baseline_min  temp_diff_from_baseline_max  temp_diff_from_baseline_std  glucose_mean  glucose_min  glucose_max  glucose_std  overall_score  composition_score  revitalization_score  duration_score  nightly_temperature   duration  minutestofallasleep  minutesasleep  minutesawake  minutesafterwakeup  timeinbed  efficiency  duration_exercise  calories_exercise  heartrate_exercise activity_type  in_default_zone_3  in_default_zone_2  in_default_zone_1  below_default_zone_1  birth_year gender  ethnicity                       education sexually_active self_report_menstrual_health_literacy  age_of_first_menarche  flow_volume_score  appetite_score  exerciselevel_score  headaches_score  cramps_score  sorebreasts_score  fatigue_score  sl